# 🌾 Kaggriculture — Final Strategy and Lessons
### 从实验路线、失败与修正，到最后的两份提交

**Team: RL is all you need · Archive date: 2026-10-01 · Final result pending**

这是提交截止后的研究复盘。最终排名和奖牌尚未公布。我们保留结论的时间背景，
不用局部胜率、早期天梯分或包检查通过来代替最终比赛结果。

**English abstract.** We describe our progression from hand-written farm scheduling,
replay-based policies and reinforcement learning to a public reactive controller with
bounded local changes. The final pair combines market-order permutation and deferred
restocking in one agent, with wool-delivery priority in the other. Fresh-world paired
confirmation supports a local improvement over the mixed-order parent. It does not
establish a medal result, universal dominance, or superiority over the wool backup.
We include negative results, production costs, validation limits, and the exact final
submission source with original licenses and attribution. The original archives are linked from GitHub Release.

**Run All** 会重画冻结结果、演示计分口径、核验并导出最终源码与提交包。
无需联网、GPU、Kaggle 密钥或游戏引擎。原始大规模锦标赛不会在这里重跑。

**阅读顺序**：比赛问题 → 最终方案 → 尝试过的路线 → 确认实验 → 反例与工程 → 双槽选择 → 下载与复现。

[GitHub 仓库](https://github.com/jwj1342/Kaggriculture) ·
[比赛](https://www.kaggle.com/competitions/kaggriculture) ·
[官方评分说明](https://www.kaggle.com/competitions/kaggriculture/overview/evaluation)


## 1 · 问题：共享市场中的农场经营

双方各经营一座农场，一季 720 回合。我们要同时决定雇工、路线、种植、喂养、采集、
交仓、采购和销售。对手会改变共享市场的库存与价格；我们的动作也会改变对手随后的选择。

现金只在单局内决定胜负。跨对局计分采用胜负结果，平局各计半分；最终赛事用
Bradley–Terry 整体拟合，队伍取两个最终 agent **整体评分中的较高者**。
这要求每份 agent 自己完成整个赛事；没有逐局选择赢家的机会。
[官方解释](https://www.kaggle.com/competitions/kaggriculture/discussion/739410)。

提交截止为 **2026-09-30 23:59 UTC**。我们在截止后 00:00:56 UTC 回读官方队伍状态，
两份最终提交均为 COMPLETE。继续比赛和最终排名由主办方产生。
[官方时间线](https://www.kaggle.com/competitions/kaggriculture/overview/timeline)。


In [1]:
from pathlib import Path
import base64
import hashlib
import io
import json
import lzma
import tarfile
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

OUT = Path('postmortem_artifacts')
OUT.mkdir(exist_ok=True)


In [2]:
EVIDENCE = json.loads('{"as_of_utc": "2026-10-01T00:00:56Z", "result_status": "Final ranking and medals pending", "engine": "kaggle-environments 1.32.7", "python": "3.11", "final_active": [{"name": "mixed-deferred", "submission_id": 56720412, "uploaded_utc": "2026-09-30T22:39:43Z", "entry_point": "deferred_capacity_agent", "directory": "submissions/2026-09-30-mixed-deferred", "files": {"main.py": "4fed6bcffb090ca3ccf304ede0317ea89f3588c51deba207e29226edd511b6e7", "submission.tar.gz": "bcd7aa25609cc376bf6da063ea035771a254cc29e968a060af98fba0d024e9a5", "LICENSE.txt": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30", "NOTICE.txt": "7edddb39907fc984fb0d31d813c09d14b053de146beb2e8b492fe54148fe359b", "REVIEW.md": "b26f1da03f36f1494219d7cb7dfeaba014542ef05a3ab0215f5c163ad5cfb5a1"}}, {"name": "wool-priority", "submission_id": 56721680, "uploaded_utc": "2026-09-30T23:36:43Z", "entry_point": "wool_delivery_priority_agent", "directory": "submissions/2026-09-30-wool-priority", "files": {"main.py": "037aead27341d4c232ff0b41eac8c0139fc171c8f176008c4ddef4ae24b84ae7", "submission.tar.gz": "02df2b147649e8b537b3233876251372503af86b7b6c50d5c0d4b2d8eda83952", "LICENSE.txt": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30", "NOTICE.txt": "d841ae1266b5efa5901767aeb3928a07cc6851073940927ebc6740dc537121e8", "REVIEW.md": "2fbc6269f83c069481f28eac7aee6884615bead1ad7a9ce32d3cac46c8c8d003"}}], "confirmation": {"mixed-deferred": {"games": 419840, "worlds": 1024, "seeds": [3000000, 3001023], "hash_blocks": [0, 4], "confidence": 0.975, "bootstrap_draws": 6000}, "wool-priority": {"games": 77824, "worlds": 1024, "seeds": [2300000, 2301023], "hash_blocks": [0, 4], "confidence": 0.95}}, "effects": [{"comparison": "Combination - funded mixed", "group": "Seven modern implementation clusters", "mean_pp": 2.65067, "low_pp": 1.95304, "high_pp": 3.40411, "confidence": 0.975}, {"comparison": "Combination - deferred", "group": "Seven modern implementation clusters", "mean_pp": 0.18834, "low_pp": 0.07673, "high_pp": 0.3139, "confidence": 0.975}, {"comparison": "Combination - wool", "group": "Seven modern implementation clusters", "mean_pp": 0.7394, "low_pp": -0.05589, "high_pp": 1.55561, "confidence": 0.975}, {"comparison": "Wool - guarded sales", "group": "cha22 / funding / shepherd", "mean_pp": 3.092, "low_pp": 2.343, "high_pp": 3.939, "confidence": 0.95}], "direct_matches": [{"candidate": "Combination", "opponent": "Funded mixed", "wins": 554, "ties": 1434, "losses": 60}, {"candidate": "Wool priority", "opponent": "Guarded sales", "wins": 212, "ties": 1754, "losses": 82}], "provenance": {"data/endgame-final/FINAL_AUDIT.json": "95d9154e365cb4b5251cb005b1ff721be938ce2ba13e598c0698ba8f8254cdff", "data/endgame-20260930-mixed-deferred-confirm/PREREGISTRATION.json": "19111a6b4aa12573d031225f792f2f99637ecc2c26eb47e39dd7ceecf692c305", "data/endgame-20260930-wool-confirm/PREREGISTRATION.json": "e5d9bd35638e6e487ae68bdc07d6bdce9ee399854c9dc89a6c15b07de48e13cb"}, "limitations": ["Effects are local paired score changes; they do not predict final Kaggle ratings or medals.", "Both seats and hash blocks reuse worlds. Independent N is 1024 worlds for each confirmation, not the number of games.", "Opponent implementation clusters share upstream ancestry.", "Confidence intervals are transcribed from frozen audits, not recomputed from aggregate counts.", "Final second-slot selection used post-hoc descriptive evidence, not a new preregistered superiority test."]}')
display(pd.DataFrame([{
    'agent': x['name'], 'submission_id': x['submission_id'],
    'uploaded_utc': x['uploaded_utc'], 'entry_point': x['entry_point']
} for x in EVIDENCE['final_active']]))


,agent,submission_id,uploaded_utc,entry_point
0,mixed-deferred,56720412,2026-09-30T22:39:43Z,deferred_capacity_agent
1,wool-priority,56721680,2026-09-30T23:36:43Z,wool_delivery_priority_agent


## 2 · 最终方案：保留继承策略，单独验证新增控制层

最终生产与路线策略大量继承自社区公开的响应式控制器。我们没有重新训练这些上游路线，
也不把公开策略的能力归为本地新增层的成果。完整作者与变化说明随下载文件保留。

| 层 | 作用 | 约束与边界 |
| :--- | :--- | :--- |
| 继承的路线、生产与市场策略 | 根据观测执行原有开局、动物/作物、运输与交易计划 | 含既有适配、估值与回退逻辑 |
| Guarded sales | 在保守条件下重排纯 SELL 队列 | 保留原数量与物理动作；对不可靠投影退出 |
| Funded mixed | 排列 2–6 条完全融资、完全成交的混合买卖，最多 720 种 | 保留这一层的数量、物理动作和投影终仓 |
| Deferred restock | 延后符合条件的夜间资源采购，并按后续观察确认的额度恢复 | 外层可以改变资源数量；需核对债务、保护库存和生产后果 |
| Wool priority | 目标工人携毛时暂时屏蔽可选采肥标记，让交仓优先 | 调用父策略一次；减少采肥可能影响后续施肥 |

**主要方案**：继承策略 → guarded → funded mixed → deferred。
顺序固定，没有在看到确认结果后交换层次或重新调参。

**第二方案**：继承策略 → guarded，并加羊毛交仓优先。
它不包含 funded mixed / deferred 两层，因此保留不同的市场和资源响应。

局部不改物理动作，也仍会通过现金、价格和库存改变未来物理行为。
审查必须覆盖完整对局，不能只检查当前回合是否满足约束。


## 3 · 尝试过的路线与踩过的坑


整理于 **2026-10-01**。本文解释走过哪些路、为什么转向，以及哪些结论后来被撤回。历史分数只表示当时读数；最终两槽的执行记录见 [RUNS.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/RUNS.md)，最终 BT 排名与奖牌仍须等待官方结果。

### 怎样读这些证据

- **引擎时期分开看。** 8 月上旬的主要评测属于 1.32.6 时期；8 月 18–20 日曾出现训练仍用 1.32.6、评估已用 1.32.7 的混用。9 月冲刺按 1.32.7 冻结引擎及源码。旧收益不能直接移植到新经济。
- **后来的更正优先，但不删除原记录。** 下文同时指出原假设与撤回原因，不能把历史文档里的“死路”“天花板”读成数学定理。
- **三个层次分开验收：** 实现是否按设计工作；在固定场地是否改善；真实平台是否运行并取得竞争收益。前一层通过不代替后一层。

### 1. 原子策略与在线调度：改进了自己的引擎，却没有覆盖真实竞争

**日期 / 引擎：** 2026-08-08–14，1.32.6 时期。

- **假设与尝试：** 将土地、雇工、作物、市场、对手响应等拆成正交选项，批量生成策略；随后试了紧急任务优先、固定分区、黏住目标、提前走位等 11 种调度方案。
- **结果 / 停止原因：** 自建场里 `enhanced` 对 `barnyard` 是 384/384；真实回放中的胜率却约 49% 对 47%。不少调度变化增加了走路；该版本的贪心调度在已试方案里最好。生成器保留为可控对手和机制实验工具，主攻路线转向完整生产计划。
- **坑与修正：** 最初库中没有有效施肥路径，买动物与卖饲料又使用不一致的储备规则，导致排名部分反映实现缺陷。修复后同样 12 个种子的 `berryherd` 从 8% 到 100%，不能继续引用修复前的作物优劣。**正确的规则推导也必须做双边 A/B：市场竞争会改变单农场最优解。**

证据：[真实场地与引擎修复 406e895](https://github.com/jwj1342/Kaggriculture/commit/406e895)、[11 种调度器 d360bec](https://github.com/jwj1342/Kaggriculture/commit/d360bec)。旧路线汇总在归档 `docs/ROADMAP.md` §11。

### 2. 真实轨迹、重放与混合接管：先证明重放的是同一局

**日期 / 引擎：** 2026-08-10–14，1.32.6；后来又采集了 1.32.7 轨迹。

- **假设与尝试：** 从强队回放提取完整季节动作，作为对手、可复用计划，或只接管开局后交回自己的调度器；再叠市场适配层。
- **结果 / 停止原因：** 裸动作序列会因杂草、资源和对手变化而执行失败；增加回放比例曾持续优于提前交回自己的调度器。项目最终保留“计划 + 响应观测的包装层”，没有将回放本身当成原选手的完整策略。
- **坑与修正：** 回放曾错一帧：`steps[i].action` 产生的是该条观察，不能拿它作当前观察之后的动作。错误重放仍能赚到原局 114% 的钱，因而曾被误判成功。修正验收为原双方资金、动作时序及商店序列一致；此前基于错位回放的若干“严格可传递”“迁移损失”结论作废。

证据：[错位修复与重测 626d042](https://github.com/jwj1342/Kaggriculture/commit/626d042)、[回放验证纪律 75b2013](https://github.com/jwj1342/Kaggriculture/commit/75b2013)、[首次补入 1.32.7 轨迹 9907965](https://github.com/jwj1342/Kaggriculture/commit/9907965)。

### 3. 市场干扰、抢跑与按商店选计划：有效范围比故事小

**日期 / 引擎：** 第一轮 2026-08-13 / 1.32.6；再次研究 09-04–05 / 1.32.7。

- **假设与尝试：** 倾销、囤积、抢先出售、切片卖单、按商店切换计划，分别试图压低对手收入或提高自身售价。
- **结果 / 停止原因：** 8 月的 19 条干扰臂都使对手更富：减少本方持续销售，也撤掉了原有价格压力。抢跑在镜像中很大，在若干非镜像回放中逐局无效；该轮卖单排序仍有收益。9 月自然抽签复验又否定了强制商店环境中“换计划多赚约 1.9 万”的结论。
- **坑与修正：** 市场按双方同一订单槽结算，不能把“同回合”当成“同槽位”；自己的收入增加也可能同时养好对手。后续报告按低、中、高同槽争抢率分层，并使用完整活策略对抗。有限计划库里的小收益，只能限制那个选择器；**它不是任意动态农场或更细分支计划的上界。**

证据：[ANALYSIS.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/ANALYSIS.md)（标注为 8 月历史研究）、[人工抽签造成假收益 770d7ba](https://github.com/jwj1342/Kaggriculture/commit/770d7ba)、[争抢场地更正 be11a85](https://github.com/jwj1342/Kaggriculture/commit/be11a85)。9 月最终排序改动经过新引擎、新对手确认，不能拿 8 月“路线关闭”直接否决它。

### 4. 测量尺子数次反转：样本量解决不了目标选错

**日期 / 引擎：** 2026-08-13–09-08，跨两个引擎时期。

- **假设与尝试：** 先用录制局收入挑计划，再用十对手胜率、直接对打、工人数、扩大面板、BT / Nash 等聚合方式校准强弱。
- **结果 / 停止原因：** `best_score` 很大部分反映那一局有多富；十个强对手的面板又在低分 RL 区间几乎全为零。9 月 2 日扩大至 20 对手后区分力改善；同批数据的 BT 与普通得分率排序相同，改善来自对手覆盖，不能归功于换估计量。
- **坑与修正：** k06 相对 k01 的早期线上优势曾被宣布为本地尺子的前瞻成功，8 月 23 日又按后续读数宣布反转；**9 月 4 日再次检查发现两者仍不符合当时内部稳定规则，因此这对“已知答案”的资格也撤销了。** 不能只截取中间一次判词证明所有本地评测无效。工人数与成绩相关也不证明强制多雇工有效；钱、差额和固定面板得分均不能直接换算奖牌。

证据：[反转记录 87979a2](https://github.com/jwj1342/Kaggriculture/commit/87979a2)、[扩大面板而非改 BT：9ffed96](https://github.com/jwj1342/Kaggriculture/commit/9ffed96)、[撤销已知答案 30af7b1](https://github.com/jwj1342/Kaggriculture/commit/30af7b1)、[撤回“补钱翻败局→榜分”4e36a3f](https://github.com/jwj1342/Kaggriculture/commit/4e36a3f)。当时“败率达到三分之一”是项目内部经验规则，不是官方 BT 收敛定理。

### 5. 完整 RL：训练工程跑通，竞争目标仍未达到

**日期 / 引擎：** 2026-08-14–09-08；8 月 20 日前后的版本漂移见第 9 节。

- **假设与尝试：** 用 PPO 读取棋盘学习长期生产，逐步加入 BC 初始化、课程、自博弈、残差、逐工人动作头、FEED、前瞻奖励和市场头。实现了 CPU 移植、张量引擎及 TorchRL 统一训练；训练加速与规则等价分别验收。
- **结果 / 停止原因：** 第一代约 1.2 亿步后在当时花名册 4/10 的水平收口；后续 `future-mkt` 在扩展本地面板优于早期纯 RL，但对四个强锚点仍是 0/3,072。9 月 29 日归档，截止前资源转向已有可用计划上的窄改动。结论是**这套已试表示、数据和训练流程没有达到目标**，并非证明 RL 无法解决比赛。
- **坑与修正：** 吞吐提高、对 starter 全胜、模型容量加大都不能代替强对手收益。单回合一个市场宏无法表达教师常见的多订单；逐工人头存在，也不代表已经学到可持续的跨回合取货、种植和照料链。

证据：[第一代复盘 6bde570](https://github.com/jwj1342/Kaggriculture/commit/6bde570)、[张量训练通过阶段门 b349ae5](https://github.com/jwj1342/Kaggriculture/commit/b349ae5)、[强锚点结果 ab58654](https://github.com/jwj1342/Kaggriculture/commit/ab58654)、[归档 7a5e91e](https://github.com/jwj1342/Kaggriculture/commit/7a5e91e)。完整结构和阶段判词保存在归档 `rl/README.md`、`rl/TODO.md`。

### 6. 模仿、接力与奖励塑形：瓶颈判断必须跟着反例修正

**日期 / 引擎：** 2026-08-20–09-08，主要为 1.32.7。

- **假设与尝试：** 教师交叉熵、回放前缀后接管、反向课程、提高长期信用、增加探索、补资产价值，试图教会稀有的购地购畜决策和长期执行。
- **结果 / 停止原因：** 单条 712 回合教师只有两次 `BUY_LAND`；放大损失权重不能补出缺少的状态覆盖。接力网络在自己的训练分布中可能改善，换进另一个混合体却退步。固定农场只交给网络市场决策时，原先忘记嫁接买种等采购，第一轮诊断就发现农场无法生产。
- **坑与修正：** 奖励曾按现价估值，市场被对手压价时反而惩罚生产；现金、作物和动物的信用折扣也可能奖励提前套现。相反，给仓内动物无上限记信用又会鼓励囤积。修正需同时检查实际行为和完整局收益。旧记录中某些 `longcredit` 负数后来因配对重算改为未定，不能作为整个信用分配方法族已失败的独立证明。

证据：[稀有标签不足 fd6b098](https://github.com/jwj1342/Kaggriculture/commit/fd6b098)、[接力不可直接组合 41cbd47](https://github.com/jwj1342/Kaggriculture/commit/41cbd47)、[必须同时嫁接采购 a302ec1](https://github.com/jwj1342/Kaggriculture/commit/a302ec1)、[现价塑形故障 ab72d3a](https://github.com/jwj1342/Kaggriculture/commit/ab72d3a)、[动物信用封顶 b65e451](https://github.com/jwj1342/Kaggriculture/commit/b65e451)。配对重算更正在归档 `docs/RUNS.md` 开头总览及后续原始判词。

### 7. RL 市场残差与真实提交：绝不能写成“RL 从未提交”

**日期 / 引擎：** 2026-08-21–09-08，1.32.7 时期。

- **假设与尝试：** 固定已有农场，训练多槽市场残差；导出时继续嫁接同一农场与采购路径。另有纯 RL 和“前 12 天回放、后续 RL”提交。
- **结果 / 停止原因：** 8 月 21 日的 `55668491`、`55673426`、`55676605` 属于 RL 提交，8 月 22 日 `55688747` 是混合体。9 月 7 日 `residord4`（`56081484`）在包检查通过、性能门未通过的情况下，经用户知情明示提交。9 月 8 日 `56099838` 也实际提交；其早期读数不能当稳定奖牌证据。
- **坑与修正：** 第 0 个市场槽漏了 NOOP 先验，修补后训练和未训练网络在该固定探针上都变成同样的 +261，不能把改善归给训练。还发现固定探针对应的 `best.pt` 不保存、自博弈快照漏嫁接，以及比较的两个路径实际是同一文件。导出、训练、评估、自博弈必须分别证明执行的是同一策略。

证据：[历史提交清点 395d697](https://github.com/jwj1342/Kaggriculture/commit/395d697)、[56081484 的授权、负门与实际提交 040e786](https://github.com/jwj1342/Kaggriculture/commit/040e786)、[NOOP / 快照 / 同文件问题 ca85752](https://github.com/jwj1342/Kaggriculture/commit/ca85752)、[56099838 提交 d460583](https://github.com/jwj1342/Kaggriculture/commit/d460583)。赛后整理时发现 `CLAUDE.md` 仍残留“RL 从未提交”的旧说明，已按这些记录更正。

### 8. 导出与评估身份：没有报错也可能跑错策略

**日期 / 引擎：** 整个开发期；属于框架和工具问题。

- **假设与尝试：** 将训练或包装策略导出为 `.py` / tar，复用进程池快速评估。
- **结果 / 停止原因：** 同名模块缓存曾使第二个策略拿到第一个的权重；多个 `main.py` 按 basename 入库又合并为同一选手。Kaggle 的 `get_last_callable` 还可能选到后来绑定的旧函数，即使变量 `agent` 指向新版。
- **坑与修正：** 用完整源码身份和独立模块名区分选手；实际解包后走官方 loader，核最终 callable、权重和活跃层。除了无异常完整运行，还检查本该触发的机制确实触发、双方动作及资金是否一致。全零差异首先排查同文件与旁路，不急着宣布机制无价值。

证据：[真实 loader 修复 12817b2](https://github.com/jwj1342/Kaggriculture/commit/12817b2)、[模块缓存 ed27502](https://github.com/jwj1342/Kaggriculture/commit/ed27502)、[basename 合并与标定 1e55a12](https://github.com/jwj1342/Kaggriculture/commit/1e55a12)、[VALIDATING.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/VALIDATING.md)。

### 9. 引擎漂移：与自己的副本一致还不够

**日期 / 引擎：** 2026-08-18–20，1.32.6 → 1.32.7。

- **假设与尝试：** NumPy、张量训练引擎逐字节匹配项目维护的规则副本，便认为与正式评估一致。
- **结果 / 停止原因：** 重建环境拉入 1.32.7，但项目的规则、NumPy 引擎和参考文件仍是 1.32.6；训练旧经济、评估新经济持续约两天。胡萝卜、番茄、鸡蛋的稀缺侧价格函数已改变。
- **坑与修正：** 内部等价测试共同引用旧副本，全部通过也发现不了外部漂移。增加直接针对已安装官方包的 2,022 个价格点校验，并同步各镜像和锁文件；旧提交保留原字节、标记时期，不能为了“统一”改写历史。

证据：[完整事故与修复 b456b2b](https://github.com/jwj1342/Kaggriculture/commit/b456b2b)。9 月底重新评估历史高分文件也是同一原则：旧文件曾读到 2302，不等于它在最终引擎和对手场仍具备那种相对位置。

### 10. 集群与数据：作业结束不等于实验完成

**日期：** 2026-08–09；不依赖游戏版本。

| 假设与尝试 | 实际结果 / 停止原因 | 坑与修正 |
|---|---|---|
| 多个 array 任务直接登记 SQLite | 并发写曾损坏数据库，后来完整恢复 | 分片只写 JSONL；受控单写者入库，核唯一键、行数与完整性。[406e895](https://github.com/jwj1342/Kaggriculture/commit/406e895) |
| 分片产生后人工记得入库 | 8 月 14 日发现 44 个目录、2,099,325 局未入库；9 月又复发 | 把 `status → ingest → 数量核对 → 导出` 作为交付链；只在确认入库一致后删分片。[1eeca7f](https://github.com/jwj1342/Kaggriculture/commit/1eeca7f) |
| 用整个 commit 是否变化保护排队实验 | 仅追加文档就使待跑链失败 | 冻结实际训练依赖和源码；文档更新不能冒充策略漂移。[e6ae614](https://github.com/jwj1342/Kaggriculture/commit/e6ae614) |
| `COMPLETED` 说明续训做了工作 | 默认 `--iters=10` 已在 pilot 耗尽，后续链零迭代仍返回成功 | 核新增训练步、CSV、checkpoint 和停止原因；显式登记预算。[fb915a5](https://github.com/jwj1342/Kaggriculture/commit/fb915a5) |
| 提交脚本能看到登录节点的临时文件 | `/tmp` 是节点本地，计算节点读不到依赖 | 所有输入放共享路径；`sbatch` 复制脚本并不复制它引用的文件。[a9e8ac5](https://github.com/jwj1342/Kaggriculture/commit/a9e8ac5) |
| checkpoint 覆盖写与 walltime 退出安全 | 中途被杀可能留下损坏的 `latest.pt` | 原子保存，保留可验证旧 checkpoint。[defa9ea](https://github.com/jwj1342/Kaggriculture/commit/defa9ea) |

9 月底继续采用短 Slurm array、登录节点联网、作业内批量评测和带锁入库。正式确认最后一片权威 TIMEOUT 后，逐行审计发现 3,280 局已全部写完，因此保存原字节并完成封存，**没有按输赢重跑或删除局**。后续慢回合复验也保留原峰值；单核重放正常只能解释已测批次，不能保证任意平台的时间上界。见 [ENDGAME_EXECUTION.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/ENDGAME_EXECUTION.md)。

### 11. 截止前的窄机制：知道为什么失败，也要接受停止

**日期 / 引擎：** 2026-09-29–30，1.32.7。以下只概括路线；原假设、源码哈希、样本和审查均在 [ENDGAME_EXECUTION.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/ENDGAME_EXECUTION.md)。

| 假设与尝试 | 结果 / 停止原因 | 坑与修正 |
|---|---|---|
| 补上被杂草打断的牧场建造与放牛 | 整局改善小，成为已审备选，未上传 | 建一块地会改变共享 RNG、后续商店和饲料消耗；单局多一头牛不是隔离收入增量 |
| 日末少买输入，给回仓产品留空间 | 原方案遗漏次日资源 SELL；严格版主要优效门失败，停止 | `sum(PICKUP)` 不是完整需求。新延期方案另建实际成交和下一观察确认的还款账，不能悄悄放宽旧门复用失败版本 |
| 提前补肥，避免原生路线取肥失败 | 合同成立，探索没有所需胜负收益，停止 | 修好一个局部短缺不必然提高得分；不因机制正确追加大确认 |
| 用购牛替代唯一绑定的购羊，提高产值 | 可实现绑定和估值，但新世界探索七簇下降约 4.07 个百分点，停止 | 当前商店、计划照料和未来出售只是估值假设；活对手结果否定了该候选，不调阈值追结果 |
| 羊毛优先交仓；混合买卖排序；日末延期补货 | 分别探索、确认，再对组合单独验收；最终保留组合与羊毛两槽 | 提早变现、全额成交、债务归零都不等于全季生产无损；价格反馈仍可改变父策略的喂食、种植和工人路线 |

这些停止判词只针对冻结候选及预先确定的实验条件，不意味着“储肥”“购牛”“仓容”问题永久解决或没有更好设计。

### 12. 提交与最终方案：交付完成，奖牌结论仍未完成

**日期 / 引擎：** 2026-09-30 截止，1.32.7。

- **假设与尝试：** 本地门通过、独立审查实际包后再提交；在线读回并复验验证局、首两场公开局和首败。最终官方截止后活跃两槽为组合 `56720412` 与羊毛 `56721680`，9 月 30 日用了 4/5 次。详见 [RUNS.md](https://github.com/jwj1342/Kaggriculture/blob/main/docs/RUNS.md)。
- **结果与槽位取舍：** 官方规则是取两份代理各自整体 BT 中较高者，不能逐对手择优。重新核对规则后，羊毛替换了 mixed：正式面板里 mixed 对 combo 没有更高得分格，羊毛有不同响应，但只有 cha22 的对手均值偏向羊毛。这是事后备份选择，不是新的预登记 BT 优效结论。原独立审查和源码保持不变。
- **坑与修正：** 最新两份活跃，上传会自动挤掉旧版本；花掉第四次后只剩一次，无法再用两次连发完全淘汰一个最新故障版本。这项真实代价须与差异化备份一起评估。线上 COMPLETE、早期连胜和较高的暂时评分都不等于最终稳健排名；历史短期读数曾在文件未改时大幅漂移。

最终方案还必须保留以下反方边界：

1. **统计单位是世界。** 两席、同源对手、两个 `PYTHONHASHSEED` 块存在相关性；不能把 419,840 局写成同样数量的独立样本。七实现簇也不是七条互不相关的策略血统。
2. **局部合同有明确边界。** 混合排序保持当前订单成交，不保证未来种植与取麦路线不失配；延期还款完成，不保证父策略之后不因价格变化主动停止喂食。
3. **有真实代价。** 羊毛版审查保留新增供肥失败与减产；组合也有减产、收入和差额负尾。通过的含义是代价已量化并由独立审查接受，不是零副作用。
4. **覆盖有限。** 活对手、镜像、逐席、负尾、实际包和慢局复验提高了可信度，仍不能枚举未知最终对手及平台行为。单场败局中对手多卖番茄，也不能证明把本方换成番茄就能赢。
5. **最终评分尚待官方产生。** 本轮能确认的是提交与验证链完成，不能写“保住银牌”或“拿到金牌”。

证据：[组合独立 REVIEW](https://github.com/jwj1342/Kaggriculture/blob/main/submissions/2026-09-30-mixed-deferred/REVIEW.md)、[羊毛独立 REVIEW](https://github.com/jwj1342/Kaggriculture/blob/main/submissions/2026-09-30-wool-priority/REVIEW.md)、[实际提交与截止读回](https://github.com/jwj1342/Kaggriculture/blob/main/docs/RUNS.md)。评分规则原文缓存于 `data/releases/kaggle-discussions-2026-09-29.json` 的讨论 739410；公开入口为 [官方讨论](https://www.kaggle.com/competitions/kaggriculture/discussion/739410)。

### 归档怎样恢复

9 月 29 日精简保留了代码历史，同时把 RL 代码、18,920 行旧实验台账及 `CEILING.md`、`ROADMAP.md`、`INFRA.md` 等放入 `data/releases/rl-line-archive-2026-09-29.tar.gz`。该本地压缩包仍存在，但 `data/` 通常不随 Git clone 分发，不能把它当读者必有的文件。

无需解包整个归档，就能从精简前的固定 commit 恢复单份可审阅副本：

```bash
git show 60169adfe316b4e4d53ca5de1ec757ed01f42836:docs/RUNS.md > /tmp/Kaggriculture-RUNS-pre-archive.md
git show 60169adfe316b4e4d53ca5de1ec757ed01f42836:docs/ROADMAP.md > /tmp/Kaggriculture-ROADMAP-pre-archive.md
git show 60169adfe316b4e4d53ca5de1ec757ed01f42836:rl/README.md > /tmp/Kaggriculture-RL-pre-archive.md
```

这里的 `/tmp` 只供当前机器阅读，不能作为 Slurm 输入。Git 恢复的是当时已跟踪内容，不自动恢复训练权重、原始回放或所有实验数据。需要运行历史实验时，另建隔离目录并按当时 manifest / 依赖检查；不要覆盖当前冻结提交。上述 commit 链接均指向具体历史版本，避免随着 README 更新丢失证据。


## 4 · 评测设计：先冻结假设，再看全新世界

最终确认使用 **kaggle-environments 1.32.7 / Python 3.11**。

1. 冻结候选源码、对手集合、种子、门槛和诊断选择规则。
2. 用会响应的策略对打，每个世界交换座位，保留完整胜平负与资金、动作摘要。
3. 组合使用全新世界 `3000000..3001023`，羊毛使用 `2300000..2301023`。
4. 分别运行 `PYTHONHASHSEED=0` 与 `4`；各自判断通过，不合并为双倍独立样本。
5. bootstrap 以世界为单位聚类，保留同一世界里的座位和对手相关性。
6. 独立审查从原始分片重算日程、缺失项、配对结果、区间和门槛。

组合确认共 **419,840 局**，1,024 个独立世界，两个 hash 块各 209,920 局；
每块有 51 项预先固定的数值门。羊毛确认共 **77,824 局**，两个 hash 块各 38,912 局，
同样只有 1,024 个独立世界。游戏数反映计算工作量，不能当作统计独立样本数。

七组现代对手先在簇内平均，再按簇等权汇总；它们是实现簇，不是七个互不相关的策略家族。
相同种子也不能完全固定外部环境：
农田状态会影响共享随机数的消耗，后续杂草和商店因而可能变化。

**下面的区间来自冻结审查。这个 Notebook 重现表格与图形，不从汇总数据重新估计区间。**


In [3]:
effects = pd.DataFrame(EVIDENCE['effects'])
display(effects)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'svg.fonttype': 'none', 'svg.hashsalt': 'kaggriculture-postmortem'})
fig, axes = plt.subplots(2, 1, figsize=(11.8, 4.6),
                         gridspec_kw={'height_ratios': [3, 1.2]}, layout='constrained')
for ax, rows, color, title in [
    (axes[0], effects.iloc[:3], '#2D6A4F', 'Combination confirmation | seven implementation clusters | 97.5% CI'),
    (axes[1], effects.iloc[3:], '#B57728', 'Wool confirmation | three other modern opponents | 95% CI')]:
    for y, (_, row) in enumerate(rows.iterrows()):
        ax.errorbar(row.mean_pp, y,
                    xerr=[[row.mean_pp-row.low_pp], [row.high_pp-row.mean_pp]],
                    fmt='o', color=color, capsize=5, markersize=7, linewidth=2)
        ax.text(1.02, y, f'{row.mean_pp:+.3f} [{row.low_pp:+.3f}, {row.high_pp:+.3f}]',
                transform=ax.get_yaxis_transform(), va='center', ha='left', fontsize=9)
    ax.set_yticks(range(len(rows)), rows.comparison)
    ax.invert_yaxis()
    ax.axvline(0, color='#9AA0A6', linewidth=1, linestyle='--')
    ax.set_xlim(-0.3, 4.2)
    ax.set_title(title, loc='left', fontsize=11, pad=12)
    ax.grid(axis='x', alpha=0.15)
    ax.spines[['top', 'right']].set_visible(False)
axes[1].set_xlabel('Paired score change (percentage points)')
fig.suptitle('Local gains, with their uncertainty', fontsize=17, fontweight='bold')
fig.savefig(OUT / 'confirmation-effects.svg', metadata={'Date': None})
fig.savefig(OUT / 'confirmation-effects.png', dpi=160)
effects.to_csv(OUT / 'confirmation-effects.csv', index=False)
display(fig)
plt.close(fig)


,comparison,group,mean_pp,low_pp,high_pp,confidence
0,Combination - funded mixed,Seven modern implementation clusters,2.65067,1.95304,3.40411,0.975
1,Combination - deferred,Seven modern implementation clusters,0.18834,0.07673,0.31390,0.975
2,Combination - wool,Seven modern implementation clusters,0.73940,-0.05589,1.55561,0.975
3,Wool - guarded sales,cha22 / funding / shepherd,3.09200,2.34300,3.93900,0.950


<Figure size 1180x460 with 2 Axes>

**解读**：组合相对 mixed 的七组均值提高 2.65067 pp，区间下界为正；
相对 wool 的区间跨过 0，不能声称显著优于 wool。
羊毛实验的对手组与区间水平不同，不能把两行增益当作同一把尺子。

大量近亲对局是平局。下面区分原始胜率与平局计半分的得分率，避免把 62.06% 写成胜率。


In [4]:
direct = pd.DataFrame(EVIDENCE['direct_matches'])
direct['games'] = direct[['wins', 'ties', 'losses']].sum(axis=1)
direct['win_rate_percent'] = 100 * direct.wins / direct.games
direct['score_rate_percent'] = 100 * (direct.wins + 0.5 * direct.ties) / direct.games
assert direct.games.eq(2048).all()
display(direct.round(5))
direct.to_csv(OUT / 'direct-match-score-rates.csv', index=False)


,candidate,opponent,wins,ties,losses,games,win_rate_percent,score_rate_percent
0,Combination,Funded mixed,554,1434,60,2048,27.05078,62.06055
1,Wool priority,Guarded sales,212,1754,82,2048,10.35156,53.17383


## 5 · 负例、真实代价与工程检查

### 组合方案

资源观察覆盖 **20,224 个不同完整对局 / 20,480 个视图 / 10,240 个配对格**。
3,670 个延期夜晚的 24,692 单位资源债全部恢复；样本中没有残债、过期或保护库存不足。
另对正式确认中的 126 个尾部配对、252 个完整对局追查现金与资源。

但探索中 **565 个配对格某种产量下降**，羊毛净少 1,082、小麦净少 181，
51 对有某夜溢出增加。新增未喂动物夜晚 1,325、减少 660；新增事件均追到继承的
`_r85_feed` 在携麦时因估值返回 PASS。这些经济选择仍然有真实生产代价。
正式确认中最差己方现金变化 **−1,241**，最差双方差额变化 **−935**。

### 羊毛方案

9,216 个完整资源观察局中，18 个配对新增供肥失败，来自 7 个世界。
肥料收集净少 4,017，羊毛收获净少 408，羊毛销售收入净增 305,693。
这些是整批对局的合计，不能读作每局收益，更不能当作单一机制的因果分解。
缺麦 FEED 无效动作和动物逃逸没有增加，但主动不喂的动物夜次数净增 196。
所有 20 个近亲确认回归格均保留并复演，未从统计中剔除。

### 真正上传的包也要检查

两份 tar 都通过 28 个压力配置，根目录只含 `main.py`、`LICENSE.txt`、`NOTICE.txt`。
检查官方 loader 最后绑定的 callable，分别运行 schema 开关配对，并补充会实际触发新增层的场景。
仅仅运行一局没有触发 helper 的游戏，不能证明 helper 安全。

组合的 1,222 个选定慢局、羊毛的 214 个慢局都用原 hash、单 CPU 完整复验。
候选隔离峰值分别为 0.416425 / 0.300854 秒。原并行运行中，组合候选峰值为 14.689405 秒，
组合确认全场最大值 17.5766 秒属于对手 shop0913；羊毛候选峰值为 7.657571 秒。
原始慢峰仍留在证据里。
这解释了观察到的慢峰，不保证其他硬件的普遍时间上界。

继承代码仍有 S839 回退；新层无错误不能写成整个控制器从无内部错误。
两个 hash 块的组合得分相同，但有 350 / 209,920 对局的完整动作/现金/状态指纹不同；
它们也没有因得分相同而被隐去。

**反方审查的作用**：要求追查尾部、重算结果、确认真实包身份和机制，而不只是阅读均值。
最终批准接受这些有证据的经济取舍，不等于证明全局无损。


## 6 · 为什么最后是 combo + wool

最后一天按顺序提交 guarded sales、funded mixed、mixed deferred、wool priority，共用 **4 / 5 次**。
最终两个活跃提交是 `56720412` 与 `56721680`。

我们先保留 combo + mixed，随后针对“两个整体 BT 取较高值”的官方规则重新审查第二槽。
frozen panel 上 mixed 比 combo 得分更好的配对格为 0；wool 有 **284 格 / 61 个世界**。
wool 相对 combo 在 cha22 上的均值为 +2.05078 pp，名义 97.5% 区间为 [0.34180, 3.80920] pp。

**这是事后描述性证据**，不是经过多重比较调整的新优效结论。
wool 的七组现代均值仍低约 0.7394 pp，在部分近亲与 Harvest 上也更差。
最终场地权重未知，那种差异可能没有实际价值；替换 mixed 也会失去它在未测场景下的保障。

更换第二槽是一项不确定性下的选择。不能把 284 格相加为额外胜场，
也不能把两个 agent 在每个对手上的最好表现拼成一个实际不存在的策略。
我们保留已完成线上健康检查的 combo，并上传原封不动、此前已通过审查的 wool 包。

第四次上传后只剩一次额度，无法靠再发两份彻底挤掉最新故障版本。
上传后的验证局、首两场公开局和首败均完成记录复验；检查正常只说明这些已查对局正常。


## 7 · 下载原始最终包与复核摘要

Kaggle 对源文件有 1 MB 上限，因此下方折叠单元嵌入经过联合压缩的两份源码与原独立审查。
解码后逐个核对 SHA-256 并导出；不会运行 agent。
原审查记录中的“待提交”或旧槽位建议属于当时状态，实际最终身份以上方提交表为准。

运行得到的 `kaggriculture-final-source.zip` 包含两份源码、许可证、NOTICE、REVIEW 与 `evidence.json`。
源文件与比赛上传文件保持一致。两份原始 tar 位于
[Release 的最终包合集](https://github.com/jwj1342/Kaggriculture/releases/download/postmortem-2026-10-01/kaggriculture-final-artifacts.zip)，
原始归档哈希也保留在摘要中。这里不包含全部原始锦标赛分片。


In [5]:
PAYLOAD = json.loads(lzma.decompress(base64.b64decode('...')))


In [6]:
checks = []
for item in EVIDENCE['final_active']:
    name = item['name']
    folder = OUT / name
    folder.mkdir(exist_ok=True)
    source_files = PAYLOAD[name]['files']
    assert set(source_files) == {'main.py', 'LICENSE.txt', 'NOTICE.txt', 'REVIEW.md'}
    for filename, text in source_files.items():
        contents = text.encode('utf-8')
        assert hashlib.sha256(contents).hexdigest() == item['files'][filename]
        (folder / filename).write_bytes(contents)
    (folder / 'SHA256SUMS').write_text(''.join(
        f'{item["files"][filename]}  {filename}\n' for filename in source_files))
    checks.append({'agent': name, 'submission_id': item['submission_id'],
                   'original_archive_sha256_reference': item['files']['submission.tar.gz'],
                   'all_source_files_verified': True})
(OUT / 'evidence.json').write_text(json.dumps(EVIDENCE, ensure_ascii=False, indent=2) + '\n')
(OUT / 'artifact-checks.json').write_text(json.dumps(checks, indent=2) + '\n')
archive_path = Path('kaggriculture-final-source.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    paths = [OUT / name for name in ('evidence.json', 'artifact-checks.json',
             'confirmation-effects.csv', 'confirmation-effects.svg',
             'confirmation-effects.png', 'direct-match-score-rates.csv')]
    paths += [OUT / item['name'] / name for item in EVIDENCE['final_active']
              for name in ('main.py', 'LICENSE.txt', 'NOTICE.txt', 'REVIEW.md', 'SHA256SUMS')]
    for path in paths:
        archive.write(path, path.as_posix())
display(pd.DataFrame(checks))
display(FileLink(str(archive_path)))
for item in EVIDENCE['final_active']:
    display(FileLink(str(OUT / item['name'] / 'main.py')))


,agent,submission_id,original_archive_sha256_reference,all_source_files_verified
0,mixed-deferred,56720412,bcd7aa25609cc376bf6da063ea035771a254cc29e968a0...,True
1,wool-priority,56721680,02df2b147649e8b537b3233876251372503af86b7b6c50...,True


/scratch/jwj/Kaggriculture/data/postmortem-notebook/kaggriculture-final-source.zip

/scratch/jwj/Kaggriculture/data/postmortem-notebook/postmortem_artifacts/mixed-deferred/main.py

/scratch/jwj/Kaggriculture/data/postmortem-notebook/postmortem_artifacts/wool-priority/main.py

## 8 · 复现边界与后续研究

| 层次 | 这里能复现什么 | 额外需要什么 |
| :--- | :--- | :--- |
| 本 Notebook | 表格、冻结 CI 图、胜平负计分、最终包身份与下载 | 常规 CPU Python 环境 |
| 单局策略行为 | 冻结源码在指定引擎上的完整对局 | kaggle-environments 1.32.7、Python 3.11、指定种子/对手/hash |
| 原正式确认 | 全日程、逐世界统计、置信区间与审查门 | 原 manifest、全部对手、原始分片、统计脚本与足够 CPU |
| 最终赛事 | 主办方的最终评分 | 平台继续比赛及发布结果 |

仓库 `tools/` 提供生成、评测、追踪和数据库工具；`slurm/` 提供作业入口；
`submissions/` 保存按日期冻结的策略；`docs/` 保存协议、执行记录与历史修正。
大型数据库和远端 D1 的归档/恢复位置见 GitHub README 的“数据归档”。

若继续研究，最有价值的工作是：扩大独立策略家族的对手覆盖；把供肥、喂养、运输和仓容
联合起来优化；在早期先校准本地指标与线上读数；保留未来数据做最终确认。
这些是后续假设，目前没有新的实验结果支持它们。

### 来源与致谢

最终控制器继承了 Thomas Tschinkel、Yusuke Hayashi、aurax7、Ahmed Berat Ozer、
shiiin9、Dmitrii Gluzdov 等社区作者的公开代码与路线。
两份包的 Apache-2.0 文本、源码注释和 NOTICE 完整保留；NOTICE 也保留上游路线数据的溯源限制。
我们的本地工作是新增控制层、实验与审查，没有补造上游路线的训练或采集经历。

- [直接父控制器来源](https://www.kaggle.com/code/shiiin9/your-market-list-is-an-order-book)
- [V56 输入预算](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v56-smarter-seeds-and-fertilizer)
- [公共 State Router](https://www.kaggle.com/code/thomastschinkel/kaggriculture-93-8-win-rate-public-state-router)
- [官方引擎](https://github.com/Kaggle/kaggle-environments)
- [仓库与完整复盘](https://github.com/jwj1342/Kaggriculture)

**Final result pending.** 提交完成、运行检查通过、本地实验改善，分别回答不同的问题。
最终名次与奖牌留待官方结果。
